<a href="https://colab.research.google.com/github/hernon33/Project_Board/blob/main/Epic4_Tasks_Aidan.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV

from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    roc_auc_score,
    accuracy_score,
    precision_score,
    recall_score,
    classification_report
)

# ============================
# Load dataset
# ============================

df = pd.read_csv("Initial_Modeling_Dataset.csv")

# Target

y = df["attrition_flag"]

X = df.drop(
    columns=[
        "attrition_flag",
        "time_spend_company",
        "salary_cat",
        "tenure_band_cat",
        "department_cat"
    ],
    errors="ignore"
)

# ============================
# Train / test split
# ============================

X_train, X_test, y_train, y_test = train_test_split(

    X,
    y,

    test_size=0.30,
    random_state=42,
    stratify=y
)

# ============================
# 1. Train Random Forest
# ============================

rf = RandomForestClassifier(

    n_estimators=200,

    random_state=42,

    class_weight="balanced",

    n_jobs=-1
)

rf.fit(

    X_train,
    y_train
)

pred_prob = rf.predict_proba(X_test)[:, 1]

auc = roc_auc_score(

    y_test,
    pred_prob
)

print("Random Forest AUC:", auc)

# ============================
# 2. Hyperparameter tuning
# ============================

param_grid = {

    "n_estimators": [100, 200, 300],

    "max_depth": [5, 10, None],

    "min_samples_split": [2, 5, 10],

    "min_samples_leaf": [1, 2, 4]
}

grid = GridSearchCV(

    RandomForestClassifier(
        random_state=42,
        class_weight="balanced"
    ),

    param_grid,

    cv=5,

    scoring="roc_auc",

    n_jobs=-1,

    verbose=1
)

grid.fit(

    X_train,
    y_train
)

print("Best params:", grid.best_params_)

best_model = grid.best_estimator_

# ============================
# Evaluate best model
# ============================

pred = best_model.predict(X_test)

pred_prob = best_model.predict_proba(X_test)[:, 1]

print("AUC:", roc_auc_score(y_test, pred_prob))

print("Accuracy:", accuracy_score(y_test, pred))

print("Precision:", precision_score(y_test, pred))

print("Recall:", recall_score(y_test, pred))

print(classification_report(y_test, pred))

Random Forest AUC: 0.9925660163939203
Fitting 5 folds for each of 81 candidates, totalling 405 fits
Best params: {'max_depth': None, 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 200}
AUC: 0.9925660163939203
Accuracy: 0.9893333333333333
Precision: 0.9923002887391723
Recall: 0.9626517273576097
              precision    recall  f1-score   support

           0       0.99      1.00      0.99      3429
           1       0.99      0.96      0.98      1071

    accuracy                           0.99      4500
   macro avg       0.99      0.98      0.99      4500
weighted avg       0.99      0.99      0.99      4500

